In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os
import shutil

zip_path = "/content/sample_data/fruit data set.zip"
extract_path = "/content/FRUITS DISEASES CLASSIFICATION"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['FRUITS DISEASES CLASSIFICATION']


In [ ]:
source_path = "/content/FRUITS DISEASES CLASSIFICATION/FRUITS DISEASES CLASSIFICATION"
target_path = "/content/final_dataset"

if os.path.exists(target_path):
    shutil.rmtree(target_path)

os.makedirs(target_path, exist_ok=True)

fruit_folders = ["APPLE", "GUAVA", "MANGO", "POMEGRANATE"]

for fruit in fruit_folders:
    fruit_path = os.path.join(source_path, fruit)

    for disease_folder in os.listdir(fruit_path):
        disease_path = os.path.join(fruit_path, disease_folder)

        if os.path.isdir(disease_path):
            shutil.copytree(
                disease_path,
                os.path.join(target_path, disease_folder)
            )

print("Classes:")
print(os.listdir(target_path))
print("Number of classes:", len(os.listdir(target_path)))

Classes:
['Anthracnose_Pomegranate', 'Bacterial_Blight_Pomegranate', 'Anthracnose_Mango', 'Alternaria_Mango', 'Rot_Apple', 'Anthracnose_Guava', 'Fruitfly_Guava', 'Black Mould Rot (Aspergillus)_Mango', 'Scab_Apple', 'Healthy_Mango', 'Healthy_Apple', 'Alternaria_Pomegranate', 'Stem and Rot (Lasiodiplodia)_Mango', 'Healthy_Guava', 'Healthy_Pomegranate', 'Cercospora_Pomegranate', 'Blotch_Apple']
Number of classes: 17


In [ ]:
!pip install rembg onnxruntime

In [ ]:
from rembg import remove
import cv2
import numpy as np
import os
import shutil
from tqdm import tqdm

input_dir = "/content/final_dataset"
output_dir = "/content/final_dataset_processed3"

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)

def remove_bg_black_resize(img):

    rgba = remove(img)


    if rgba.shape[2] == 4:
        alpha = rgba[:, :, 3]
        rgb = rgba[:, :, :3]

        black_bg = np.zeros_like(rgb)
        mask = alpha > 0
        black_bg[mask] = rgb[mask]

        img = black_bg
    else:
        img = rgba


    h, w = img.shape[:2]
    size = max(h, w)

    square = np.zeros((size, size, 3), dtype=np.uint8)

    y_offset = (size - h) // 2
    x_offset = (size - w) // 2

    square[y_offset:y_offset+h, x_offset:x_offset+w] = img


    final = cv2.resize(square, (224, 224))

    return final

for class_name in os.listdir(input_dir):
    class_path = os.path.join(input_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    save_class_path = os.path.join(output_dir, class_name)
    os.makedirs(save_class_path, exist_ok=True)

    for img_name in tqdm(os.listdir(class_path), desc=class_name):
        img_path = os.path.join(class_path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        processed = remove_bg_black_resize(img)

        save_path = os.path.join(save_class_path, img_name)
        cv2.imwrite(save_path, processed)

print("Done ✅")
print("Processed dataset:", output_dir)

Blotch_Apple: 100%|██████████| 100/100 [03:42<00:00,  2.22s/it]

Done ✅
Processed dataset: /content/final_dataset_processed3


In [ ]:
import zipfile
import os
import shutil

zip_path = "/content/final_dataset_processed33.zip"

extract_path = "/content/final_dataset_processed"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done ✅")
print(os.listdir(extract_path))

Done ✅
['content']


In [ ]:
import os

for root, dirs, files in os.walk("/content/final_dataset_processed"):
    print(root)
    print(dirs[:10])

    if len(dirs) > 5:
        break

/content/final_dataset_processed
['content']
/content/final_dataset_processed/content
['final_dataset_processed3']
/content/final_dataset_processed/content/final_dataset_processed3
['Anthracnose_Pomegranate', 'Bacterial_Blight_Pomegranate', 'Anthracnose_Mango', 'Alternaria_Mango', 'Rot_Apple', 'Anthracnose_Guava', 'Fruitfly_Guava', 'Black Mould Rot (Aspergillus)_Mango', 'Scab_Apple', 'Healthy_Mango']


In [ ]:
import os
import shutil
import random
source_dir = "/content/final_dataset_processed/content/final_dataset_processed3"
output_dir = "/content/splitted_dataset"
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)
classes = os.listdir(source_dir)
for class_name in classes:
    class_path = os.path.join(source_dir, class_name)
    if not os.path.isdir(class_path):
        continue
    images = os.listdir(class_path)
    random.shuffle(images)
    total = len(images)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))
    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]
    for split in ['train', 'val', 'test']:
        os.makedirs(
            os.path.join(output_dir, split, class_name),
            exist_ok=True
        )
    for img in train_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(output_dir, 'train', class_name, img)
        )
    for img in val_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(output_dir, 'val', class_name, img)
        )
    for img in test_imgs:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(output_dir, 'test', class_name, img)
        )

print("Done ✅")
print(os.listdir(output_dir))

Done ✅
['train', 'val', 'test']


In [ ]:
!zip -r splitted_dataset_dataset.zip /content/splitted_dataset

Streaming output truncated to the last 5000 lines.
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_152.jpg (deflated 4%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_085.jpg (deflated 5%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_091.jpg (deflated 4%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_114.jpg (deflated 3%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_130.jpg (deflated 3%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_135.jpg (deflated 3%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_076.jpg (deflated 4%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_105.jpg (deflated 4%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_073.jpg (deflated 4%)
  adding: content/splitted_dataset/train/Alternaria_Mango/Alternaria_057.jpg (deflated 3%)
  adding: content/splitted_dataset/trai

In [ ]:
from google.colab import files

files.download("splitted_dataset_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf

train_path = "/content/splitted_dataset/train"
val_path = "/content/splitted_dataset/val"

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    val_path,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_data.class_names

Found 4571 files belonging to 17 classes.
Found 982 files belonging to 17 classes.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.prefetch(AUTOTUNE)
val_data = val_data.prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([

    layers.Rescaling(1./255),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 61s 285ms/step - accuracy: 0.7766 - loss: 0.7231 - val_accuracy: 0.8931 - val_loss: 0.2976
Epoch 2/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.8952 - loss: 0.3169 - val_accuracy: 0.9277 - val_loss: 0.2224
Epoch 3/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.9278 - loss: 0.2225 - val_accuracy: 0.9420 - val_loss: 0.1905
Epoch 4/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.9414 - loss: 0.1772 - val_accuracy: 0.9440 - val_loss: 0.1738
Epoch 5/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.9497 - loss: 0.1511 - val_accuracy: 0.9450 - val_loss: 0.1737
Epoch 6/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.9532 - loss: 0.1313 - val_accuracy: 0.9470 - val_loss: 0.1591
Epoch 7/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.9670 - loss: 0.1062 - val_accuracy: 0.9552 - val_loss: 0.1496
Epoch 8/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.9707 - loss: 0.0888 - val_ac

In [ ]:
test_path = "/content/splitted_dataset/test"

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

test_loss, test_acc = model.evaluate(test_data)

print("Test Accuracy:", test_acc)

Found 984 files belonging to 17 classes.
31/31 ━━━━━━━━━━━━━━━━━━━━ 15s 492ms/step - accuracy: 0.9492 - loss: 0.1629
Test Accuracy: 0.9491869807243347


In [ ]:
model.save("Newfruit_disease_model_final.h5")

In [ ]:
with open("Newclasses.txt", "w") as f:
    for c in class_names:
        f.write(c + "\n")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("Newfruit_disease_model_final.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpf_vdfo8y'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 17), dtype=tf.float32, name=None)
Captures:
  140367482867152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482867536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482869840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482869456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482868880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482867344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482870032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482870800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482870416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482867728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140367482

In [ ]:
from google.colab import files

files.download("Newfruit_disease_model_final.h5")
files.download("Newfruit_disease_model_final.tflite")
files.download("Newclasses.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>